# AudioMind: Build a pyannote-Powered Conversation Intelligence Demo

This notebook shows how AudioMind uses pyannote to turn raw conversation audio into something developers can build on: a reliable speaker timeline.

A plain transcript tells you what was said, but it loses one of the most important parts of a community call: **who said it and when**. pyannote adds that missing structure. Once AudioMind knows the speaker turns, it can attach transcript text, identify known voices, and ask an LLM to extract follow-up work.

**Audience:** developers building AI workflow demos, DevRel tooling, meeting intelligence prototypes, or community automation.

**What you will build in AudioMind:**

- speaker diarization with pyannote: who spoke when
- speech-to-text with word timestamps
- speaker-attributed transcript turns
- community insights from Gemma: pain points, content ideas, and calls to action
- a hosted Precision-2 path where pyannoteAI returns diarization + transcription directly
- optional voiceprints for known-speaker identification

## Why pyannote Matters

Without pyannote, a developer would need to solve hard audio problems manually:

- detect when speech starts and stops
- separate multiple speakers
- decide when one speaker changes to another
- connect transcript words back to the right person
- handle overlaps, pauses, and messy meeting audio
- build enough structure for an LLM to produce useful follow-ups

That is a lot of low-level audio work before the product even becomes useful.

pyannote gives us the key primitive: **speaker turns**. Once we have that, the rest of the workflow becomes much simpler:

1. pyannote creates the speaker timeline.
2. Whisper provides the words.
3. A small overlap step attaches words to pyannote speaker turns.
4. Gemma turns the speaker-aware transcript into community insights.

The hosted Precision-2 section shows the even shorter version: pyannoteAI can return speaker-labeled transcription directly.


## Install Dependencies

Run this once before the local pyannote + Whisper path.

This cell uses Jupyter's `%pip` magic. It is still pip, but it installs into the current notebook kernel instead of relying on a shell-level `pip` command being available on PATH.

After installation in Colab, restart the runtime before continuing. In local Jupyter, restart the kernel if imports behave strangely.


In [ ]:
%pip install -q --upgrade pip
%pip install -q pyannote.audio==4.0.4 faster-whisper==1.0.3 transformers==4.44.2

print("OK Install done. Restart the runtime/kernel if this is the first install in this session.")


## Step 1: Configure Hugging Face Access

The local diarization model is gated. Use a read token from the same Hugging Face account that accepted the model terms.

If a later cell returns `401`, `403`, or a gated-model error, the fix is usually one of these:

- accept the model terms on Hugging Face
- use a token from the same account
- confirm the token has read access


In [ ]:
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    token = getpass("Paste your Hugging Face read token: ").strip()
    os.environ["HF_TOKEN"] = token

print("OK HF_TOKEN is set" if os.environ.get("HF_TOKEN") else "HF_TOKEN is missing")


## Step 2: Choose Audio

For a reliable live demo, start with a short sample file. Once the pipeline works, replace `AUDIO_URL` with your own direct audio URL or set `AUDIO_FILE` to a local WAV/MP3/M4A file.

What this step proves: the rest of the notebook has one clear audio input to process.


In [ ]:
import urllib.request

AUDIO_URL = "https://files.pyannote.ai/marklex1min.wav"
AUDIO_FILE = "call.wav"

urllib.request.urlretrieve(AUDIO_URL, AUDIO_FILE)

print(f"OK Downloaded sample audio to {AUDIO_FILE}")


## Step 2b: Normalize Audio for Local Models

Real meeting audio is messy. Even a valid MP3/M4A/WAV file can have sample-rate, stereo, metadata, or decoder quirks.

pyannote works on precise time chunks, so we normalize local audio before diarization. This gives pyannote a clean 16 kHz mono WAV file and avoids sample-count errors.

This is not changing the conversation. It is making the audio easier for ML models to read consistently.


In [ ]:
from pathlib import Path
import shutil
import subprocess

source_audio = Path(AUDIO_FILE)
AUDIO_FILE_FOR_MODELS = "call.16khz.mono.wav"

if shutil.which("ffmpeg") is None:
    print("ffmpeg is not installed. Using original audio file:", AUDIO_FILE)
    AUDIO_FILE_FOR_MODELS = AUDIO_FILE
else:
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i", str(source_audio),
            "-ac", "1",
            "-ar", "16000",
            AUDIO_FILE_FOR_MODELS,
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    print("OK Normalized audio for local models:", AUDIO_FILE_FOR_MODELS)


In [ ]:
from pathlib import Path
import shutil
import subprocess

source_audio = Path(AUDIO_FILE)
AUDIO_FILE_FOR_MODELS = "call.16khz.mono.wav"

if shutil.which("ffmpeg") is None:
    print("ffmpeg is not installed. Using original audio file:", AUDIO_FILE)
    AUDIO_FILE_FOR_MODELS = AUDIO_FILE
else:
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i", str(source_audio),
            "-ac", "1",
            "-ar", "16000",
            AUDIO_FILE_FOR_MODELS,
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    print("OK Normalized audio for local models:", AUDIO_FILE_FOR_MODELS)


## Step 3: Run pyannote Speaker Diarization

This is the core pyannote step.

Diarization turns one mixed audio file into a timeline of speaker segments:

`SPEAKER_00 spoke from 00:10 to 00:18`

At this stage, speakers are anonymous labels, not real names. That is expected. The important thing is that pyannote recovers the conversation structure: who spoke, when they started, and when they stopped.

Without this step, a transcript is just a wall of text. With this step, we can build a speaker-aware transcript and later analyze what each participant contributed.


In [ ]:
from pyannote.audio import Pipeline

MODEL = "pyannote/speaker-diarization-community-1"

pipeline = Pipeline.from_pretrained(MODEL, token=os.environ["HF_TOKEN"])

print("OK Loaded pyannote diarization model")


In [ ]:
diarization = pipeline(AUDIO_FILE_FOR_MODELS)

print("OK Diarization complete")


In [ ]:
diarization.speaker_diarization


In [ ]:
segments = []
for turn, speaker in diarization.speaker_diarization:
    segments.append({
        "start": round(turn.start, 2),
        "end": round(turn.end, 2),
        "speaker": speaker
    })

print(segments)


## Step 4: Transcribe with Word Timestamps

Whisper provides the words. pyannote provides the speaker timeline.

Word timestamps matter because they let us attach the transcript back to pyannote speaker turns. In other words, Whisper tells us **what was said**, and pyannote tells us **where each speaker was active**.

Sane demo defaults:

- CPU: `WhisperModel("base", device="cpu", compute_type="int8")`
- GPU: `device="cuda"`, `compute_type="float16"`


In [ ]:
from faster_whisper import WhisperModel
import torch

whisper_device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if whisper_device == "cuda" else "int8"

# Use "tiny" or "base" for quick demos. Use "small" for better quality when you can wait.
whisper = WhisperModel("base", device=whisper_device, compute_type=compute_type)

print(f"OK faster-whisper loaded on {whisper_device} with {compute_type}")


In [ ]:
whisper_segments, info = whisper.transcribe(
    AUDIO_FILE_FOR_MODELS,
    word_timestamps=True,
    vad_filter=True,
)

print(f"OK Transcription started; detected language: {info.language}")


In [ ]:
words = []

for segment in whisper_segments:
    for word in segment.words or []:
        words.append({
            "start": float(word.start),
            "end": float(word.end),
            "text": word.word.strip(),
        })

print(f"OK Extracted {len(words)} timestamped words")


In [ ]:
words


## Step 5: Attach Words to pyannote Speaker Turns

Now we combine the two model outputs.

Instead of guessing a speaker for each word in isolation, use pyannote as the source of truth for the speaker timeline. For each pyannote segment, collect the Whisper words that overlap that time range.

This is the key developer idea:

- pyannote owns the speaker structure
- Whisper owns the text
- overlap connects the two

Without pyannote, this alignment step has no reliable speaker timeline to attach words to.


In [ ]:
def overlap_seconds(start_a, end_a, start_b, end_b):
    return max(0, min(end_a, end_b) - max(start_a, start_b))


def words_for_segment(segment, words, min_overlap_ratio=0.5):
    segment_words = []

    for word in words:
        word_duration = word["end"] - word["start"]
        if word_duration <= 0:
            continue

        overlap = overlap_seconds(
            segment["start"],
            segment["end"],
            word["start"],
            word["end"],
        )

        if overlap / word_duration >= min_overlap_ratio:
            segment_words.append(word["text"])

    return " ".join(segment_words).strip()


In [ ]:
turns = []

for segment in segments:
    text = words_for_segment(segment, words)

    if text:
        turns.append(
            {
                "speaker": segment["speaker"],
                "start": segment["start"],
                "end": segment["end"],
                "text": text,
            }
        )

print(f"OK Built {len(turns)} speaker-labeled transcript turns")


In [ ]:
def mmss(seconds):
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{minutes:02d}:{secs:02d}"

for turn in turns[:6]:
    print(f'[{mmss(turn["start"])}] {turn["speaker"]}: {turn["text"]}')


In [ ]:
import json

with open("transcript.json", "w", encoding="utf-8") as f:
    json.dump(turns, f, indent=2, ensure_ascii=False)

with open("transcript.md", "w", encoding="utf-8") as f:
    for turn in turns:
        line = f'[{mmss(turn["start"])}] {turn["speaker"]}: {turn["text"]}'
        f.write(line + "\n\n")

print("OK Saved transcript.json and transcript.md")


## Step 6: Reason Over the Conversation With Gemma

Now that pyannote has given us speaker-aware structure, Gemma can do more than summarize.

The model can connect evidence to recommendations:

1. identify the moment where a speaker raised a topic, blocker, or need
2. explain the pain point or opportunity
3. suggest a content idea based on that evidence
4. give the reason why that content would help

This is where pyannote becomes product value. Because every transcript turn has a timestamp and speaker, Gemma can ground its suggestions in specific moments from the conversation instead of producing generic ideas.


In [ ]:
# One-time setup for Gemma through Ollama.
!apt-get update -qq && apt-get install -y -qq zstd
!command -v ollama >/dev/null || curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > /tmp/ollama.log 2>&1 &
!ollama pull gemma3:1b

print("OK Gemma is ready through Ollama")


In [ ]:
import json
import re
import requests

transcript = "\n".join(
    f"[{mmss(turn['start'])}] {turn['speaker']}: {turn['text']}"
    for turn in turns
)[:12000]

prompt = f"""
You are a DevRel strategist analyzing a speaker-labeled community call transcript.

Reason from the transcript. Do not only extract text.
For each recommendation, explain what the speaker said, when they said it, what pain point or opportunity it reveals, and why the suggested content would help.

Return JSON only with this exact shape:
{{
  "summary": "2 concise sentences",
  "moments": [
    {{
      "timestamp": "MM:SS",
      "speaker": "SPEAKER_00",
      "what_they_said": "short evidence from the transcript",
      "insight": "what this reveals"
    }}
  ],
  "pain_points": [
    {{
      "timestamp": "MM:SS",
      "speaker": "SPEAKER_00",
      "pain_point": "the inferred blocker or need",
      "evidence": "what the speaker said",
      "why_it_matters": "why a community or DevRel team should care"
    }}
  ],
  "content_ideas": [
    {{
      "timestamp": "MM:SS",
      "speaker": "SPEAKER_00",
      "idea": "specific content idea",
      "format": "blog | short video | docs | workshop | social post",
      "based_on": "the source moment or pain point",
      "reason": "why this content would help the audience"
    }}
  ],
  "calls_to_action": [
    {{
      "timestamp": "MM:SS",
      "speaker": "SPEAKER_00",
      "action": "recommended next step",
      "reason": "why this action follows from the transcript"
    }}
  ]
}}

Transcript:
{transcript}
"""

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "gemma3:1b",
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "format": "json",
        "options": {"temperature": 0},
    },
    timeout=600,
)
response.raise_for_status()

reply = response.json()["message"]["content"]

try:
    insights = json.loads(reply)
except json.JSONDecodeError:
    match = re.search(r"\{.*\}", reply, flags=re.DOTALL)
    if not match:
        print(reply)
        raise
    insights = json.loads(match.group(0))

print(json.dumps(insights, indent=2, ensure_ascii=False))

with open("insights.json", "w", encoding="utf-8") as f:
    json.dump(insights, f, indent=2, ensure_ascii=False)


## Run the Streamlit Demo App

The notebook explains the pipeline. The AudioMind Streamlit app turns it into a product-style demo where someone can upload audio and download results.

From the project root:

```bash
cd /Users/jyotibisht/Documents/Pyannote-demo
make setup
```

Then add your API keys to `app/.env`:

```bash
PYANNOTE_API_KEY=your_pyannoteai_key_here
HF_TOKEN=hf_your_read_token_here
```

Start the app:

```bash
make run
```

`make setup` creates the Python environment, installs dependencies, starts/checks Ollama, and pulls `gemma3:1b`. AudioMind lets a developer upload MP3/WAV/M4A audio, hear the uploaded file, inspect speaker-labeled transcript turns, identify known voices with voiceprints, and download transcript/insight JSON.


<!-- precision-2-appendix -->
## Hosted Shortcut: Precision-2 Does Diarization + Transcription

The local pipeline above is useful for teaching because developers can inspect every stage: pyannote diarization, Whisper transcription, alignment, and insight extraction.

For a production-style integration, pyannoteAI Precision-2 makes the core audio step much smaller:

> Send audio to Precision-2, request transcription, and receive speaker-labeled transcript turns.

That means pyannoteAI handles the speaker timeline and hosted speech-to-text together. No local Whisper setup. No custom word-to-speaker alignment code. The API returns the structure developers actually want to build on.

Use this section to show the API-native version of the same idea AudioMind uses in the app.


In [ ]:
import json
import os
import time
from getpass import getpass

import requests

api_key = os.environ.get("PYANNOTE_API_KEY") or getpass("Paste your pyannoteAI API key: ").strip()
audio_url = globals().get("AUDIO_URL", "https://files.pyannote.ai/marklex1min.wav")
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}


def wait_for_job(job_id):
    while True:
        response = requests.get(
            f"https://api.pyannote.ai/v1/jobs/{job_id}",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=60,
        )
        response.raise_for_status()
        job = response.json()

        print("Status:", job["status"])
        if job["status"] == "succeeded":
            return job
        if job["status"] in {"failed", "canceled"}:
            raise RuntimeError(job)
        time.sleep(5)


response = requests.post(
    "https://api.pyannote.ai/v1/diarize",
    headers=headers,
    json={"url": audio_url, "model": "precision-2", "transcription": True},
    timeout=60,
)
response.raise_for_status()

job_id = response.json()["jobId"]
print("Job:", job_id)

precision_job = wait_for_job(job_id)
precision_output = precision_job["output"]
turns = precision_output.get("turnLevelTranscription", [])

for turn in turns[:10]:
    print(f"[{mmss(turn['start'])}] {turn['speaker']}: {turn['text']}")

with open("precision2_output.json", "w", encoding="utf-8") as f:
    json.dump(precision_output, f, indent=2, ensure_ascii=False)

print("OK Saved precision2_output.json")


<!-- precision-2-appendix -->
## Bonus: Voiceprints in One Cell

Diarization gives anonymous labels like `SPEAKER_00`. Voiceprints add identity for known speakers.

For example, if you have a clean sample of Jyoti speaking, pyannoteAI can create a voiceprint and use it to identify that voice in future recordings.

This is useful when AudioMind needs names like `Host`, `Jyoti`, or `Customer` instead of anonymous speaker IDs.


In [ ]:
# Optional voiceprint demo.
# Replace this with a short clean clip where only the known speaker talks.

known_speaker_sample_url = ""  # example: "https://example.com/jyoti-voice-sample.wav"
known_speaker_name = "Jyoti"

if not known_speaker_sample_url:
    print("Add a known_speaker_sample_url to run the voiceprint demo.")
else:
    vp_job = requests.post(
        "https://api.pyannote.ai/v1/voiceprint",
        headers=headers,
        json={"url": known_speaker_sample_url, "model": "precision-2"},
    ).json()
    voiceprint = wait_for_job(vp_job["jobId"])["output"]["voiceprint"]

    identify_job = requests.post(
        "https://api.pyannote.ai/v1/identify",
        headers=headers,
        json={
            "url": audio_url,
            "model": "precision-2",
            "voiceprints": [{"label": known_speaker_name, "voiceprint": voiceprint}],
        },
    ).json()
    identified = wait_for_job(identify_job["jobId"])["output"]

    for segment in identified.get("diarization", [])[:10]:
        print(f"[{mmss(segment['start'])}] {segment['speaker']}")
